In [ ]:
!pip install xgboost shap -q

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import joblib

In [ ]:
df = pd.read_csv('/content/kidney.csv')

df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,classification
0,48,80,1.020,1,0,?,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,?,normal,notpresent,notpresent,?,...,38,6000,?,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,?,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             397 non-null    object
 1   bp              397 non-null    object
 2   sg              397 non-null    object
 3   al              397 non-null    object
 4   su              397 non-null    object
 5   rbc             397 non-null    object
 6   pc              397 non-null    object
 7   pcc             397 non-null    object
 8   ba              397 non-null    object
 9   bgr             397 non-null    object
 10  bu              397 non-null    object
 11  sc              397 non-null    object
 12  sod             397 non-null    object
 13  pot             397 non-null    object
 14  hemo            397 non-null    object
 15  pcv             397 non-null    object
 16  wbcc            397 non-null    object
 17  rbcc            397 non-null    object
 18  htn       

In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

/tmp/ipykernel_4104/2907331251.py:2: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [ ]:
for col in df.columns:
    if df[col].dtype != 'object':
        df[col] = df[col].fillna(df[col].median())

In [ ]:
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
print(df.columns)

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')


In [ ]:
target = 'classification'   # adjust if needed

X = df.drop(target, axis=1)
y = df[target]

In [ ]:
print(y.value_counts())

classification
0    248
1    149
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=8, n_estimators=300, random_state=42)

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
rf_pred = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]
xgb_prob = xgb.predict_proba(X_test)[:,1]

In [ ]:
def evaluate(name, y_true, y_pred, y_prob):
    print(f"\n==== {name} ====")
    print(classification_report(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))

In [ ]:
evaluate("Random Forest", y_test, rf_pred, rf_prob)
evaluate("XGBoost", y_test, xgb_pred, xgb_prob)


==== Random Forest ====
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        62
           1       1.00      0.97      0.99        38

    accuracy                           0.99       100
   macro avg       0.99      0.99      0.99       100
weighted avg       0.99      0.99      0.99       100

ROC-AUC: 1.0

==== XGBoost ====
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        62
           1       1.00      0.95      0.97        38

    accuracy                           0.98       100
   macro avg       0.98      0.97      0.98       100
weighted avg       0.98      0.98      0.98       100

ROC-AUC: 1.0


In [ ]:
for col in X.columns:
    print(col, X[col].nunique())

age 76
bp 10
sg 5
al 6
su 6
rbc 3
pc 3
pcc 3
ba 3
bgr 145
bu 118
sc 84
sod 34
pot 40
hemo 115
pcv 42
wbcc 89
rbcc 45
htn 3
dm 3
cad 3
appet 3
pe 3
ane 3


In [ ]:
from sklearn.model_selection import permutation_test_score

score, perm_scores, p_value = permutation_test_score(
    xgb, X, y,
    cv=5,
    scoring="roc_auc",
    n_permutations=50
)

print("Score:", score)
print("p-value:", p_value)

Score: 0.9997241379310345
p-value: 0.0196078431372549


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(xgb, X, y, cv=5, scoring='roc_auc')

print(scores)
print(scores.mean())

[1.         1.         0.99862069 1.         1.        ]
0.9997241379310345


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

scores = cross_val_score(model, X, y, cv=10, scoring='roc_auc')

print(scores.mean(), scores.std())

1.0 7.021666937153402e-17


In [ ]:
joblib.dump(model, 'kidney_model.pkl')
joblib.dump(scaler, 'kidney_scaler.pkl')

['kidney_scaler.pkl']

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def run_cv_pipeline(model, X, y, name):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring='roc_auc'
    )

    print(f"\n==== {name} ====")
    print("ROC-AUC scores:", scores)
    print("Mean ROC-AUC:", scores.mean())
    print("Std:", scores.std())

In [ ]:
run_cv_pipeline(model, X, y, "kidney")


==== kidney ====
ROC-AUC scores: [0.99966667 1.         1.         1.         1.        ]
Mean ROC-AUC: 0.9999333333333332
Std: 0.00013333333333331865
